In [ ]:
!pip install torchinfo
!pip install kaggle

In [ ]:
#download the dataset
from google.colab import files
files.upload()  # upload legacy token from kaggle when prompted

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d waseemalastal/breakhis-breast-cancer-histopathological-dataset
!unzip -q breakhis-breast-cancer-histopathological-dataset.zip

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchinfo import summary

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

from tqdm.notebook import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns

In [ ]:
data_dir = ('/content/BreakHis - Breast Cancer Histopathological Database/dataset_cancer_v1/dataset_cancer_v1/classificacao_multiclasse/100X')

class_counts = {}
for class_folder in sorted(os.listdir(data_dir)):
    class_path = os.path.join(data_dir, class_folder)
    if os.path.isdir(class_path):
        class_counts[class_folder] = len(os.listdir(class_path))

classes = np.array(list(class_counts.keys()))
counts  = np.array(list(class_counts.values()))

plt.figure(figsize=(12, 5))
sns.barplot(x=classes, y=counts, palette='viridis')
plt.title('Class Distribution - Multiclass (100X)')
plt.ylabel('Images')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('Class counts:')
for cls, cnt in class_counts.items():
    print(f'  {cls}: {cnt}')
total_samples = counts.sum()
num_classes   = len(class_counts)
class_weights = {
    cls: total_samples / (num_classes * cnt)
    for cls, cnt in class_counts.items()
}
weights_tensor = torch.tensor(
    [class_weights[c] for c in classes], dtype=torch.float32
)

print('\nClass weights:')
for cls, w in zip(classes, weights_tensor.numpy()):
    print(f'  {cls}: {w:.4f}')

In [ ]:
IMG_SIZE = 300

orig_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE))
])

train_final_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_final_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

In [ ]:
orig_dataset = datasets.ImageFolder(root=data_dir, transform=orig_transform)
class_names  = orig_dataset.classes
num_classes  = len(class_names)
print('Classes:', class_names)
print('Total images:', len(orig_dataset))
print('Num classes:', num_classes)

targets = np.array(orig_dataset.targets)

test_size         = 352
samples_per_class = test_size // num_classes
print(f'\nTest samples per class: {samples_per_class}')

class_indices = defaultdict(list)
for idx, label in enumerate(targets):
    class_indices[label].append(idx)

rng          = np.random.default_rng(seed=42)
test_indices = []
for cls, idxs in class_indices.items():
    n_take = min(samples_per_class, len(idxs))
    if n_take < samples_per_class:
        print(f'  Warning: class {class_names[cls]} only has {len(idxs)} samples, '
              f'taking {n_take} instead of {samples_per_class}')
    chosen = rng.choice(idxs, n_take, replace=False)
    test_indices.extend(chosen)

all_indices       = set(range(len(orig_dataset)))
train_val_indices = list(all_indices - set(test_indices))

test_dataset     = TransformedSubset(Subset(orig_dataset, test_indices), eval_final_transform)
train_val_subset = Subset(orig_dataset, train_val_indices)

train_size = int(0.8 * len(train_val_subset))
val_size   = len(train_val_subset) - train_size
train_raw, val_raw = random_split(
    train_val_subset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = TransformedSubset(train_raw, train_final_transform)
val_dataset   = TransformedSubset(val_raw,   eval_final_transform)

print(f'\nTrain: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}')

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        reduced = max(in_channels // reduction_ratio, 8)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, reduced),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, in_channels)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        avg_pool = F.adaptive_avg_pool2d(x, 1)   # (B, C, 1, 1)
        max_pool = F.adaptive_max_pool2d(x, 1)   # (B, C, 1, 1)
        avg_out  = self.mlp(avg_pool)             # (B, C)
        max_out  = self.mlp(max_pool)             # (B, C)
        attention = torch.sigmoid(avg_out + max_out).view(B, C, 1, 1)
        return x * attention


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=2,
            out_channels=1,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

    def forward(self, x):
        avg_out  = torch.mean(x, dim=1, keepdim=True)    # (B, 1, H, W)
        max_out, _ = torch.max(x, dim=1, keepdim=True)   # (B, 1, H, W)
        combined = torch.cat([avg_out, max_out], dim=1)  # (B, 2, H, W)
        attention = torch.sigmoid(self.conv(combined))   # (B, 1, H, W)
        return x * attention


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

In [ ]:
class EfficientNetB3WithCBAM(nn.Module):
    def __init__(self, num_classes=8, pretrained=True, reduction_ratio=16):
        super().__init__()

        weights  = models.EfficientNet_B3_Weights.DEFAULT if pretrained else None
        backbone = models.efficientnet_b3(weights=weights)
        feature_blocks = list(backbone.features.children())

        def get_out_channels(block):
            out_ch = None
            for m in block.modules():
                if isinstance(m, nn.Conv2d):
                    out_ch = m.out_channels
            return out_ch

        ch_block6 = get_out_channels(feature_blocks[6])  # 136
        ch_block7 = get_out_channels(feature_blocks[7])  # 384

        self.features = nn.Sequential(
            *feature_blocks[:6],
            feature_blocks[6],
            CBAM(ch_block6, reduction_ratio=reduction_ratio),
            feature_blocks[7],
            CBAM(ch_block7, reduction_ratio=reduction_ratio),
            feature_blocks[8]
        )

        self.avgpool = backbone.avgpool
        in_features  = 1536

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )

        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def freeze_backbone(self):
        for p in self.features.parameters():
            p.requires_grad = False
        for block in self.features:
            if isinstance(block, CBAM):
                for p in block.parameters():
                    p.requires_grad = True
        print('Backbone frozen - training classifier head + CBAM only.')

    def unfreeze_backbone(self, unfreeze_last_n_blocks=None):
        if unfreeze_last_n_blocks is None:
            for p in self.features.parameters():
                p.requires_grad = True
            print('Full backbone unfrozen - end-to-end fine-tuning.')
        else:
            blocks = list(self.features.children())
            for block in blocks[-unfreeze_last_n_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True
            print(f'Last {unfreeze_last_n_blocks} feature blocks unfrozen.')


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = EfficientNetB3WithCBAM(num_classes=num_classes, pretrained=True).to(device)
print(summary(model, input_size=(1, 3, 300, 300), verbose=0))

In [ ]:
def compute_metrics(labels, preds):
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='weighted', zero_division=0)
    rec  = recall_score(labels, preds, average='weighted', zero_division=0)
    f1   = f1_score(labels, preds, average='weighted', zero_division=0)
    cm = confusion_matrix(labels, preds)
    sensitivity_per_class, specificity_per_class = [], []
    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - (TP + FP + FN)
        sensitivity_per_class.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        specificity_per_class.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    return acc, prec, rec, f1, np.mean(sensitivity_per_class), np.mean(specificity_per_class)


def normalize_image(img_tensor):
    img = img_tensor.clone()
    return (img - img.min()) / (img.max() - img.min() + 1e-8)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    all_preds, all_labels = [], []
    running_loss = 0.0
    for images, labels in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return (running_loss / len(loader.dataset),) + compute_metrics(all_labels, all_preds)


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    all_preds, all_labels = [], []
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return (running_loss / len(loader.dataset),) + compute_metrics(all_labels, all_preds)


def test_model(model, loader, criterion, device, class_names=None):
    model.eval()
    all_preds, all_labels = [], []
    all_images, all_probs = [], []
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Testing', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_images.extend(images.cpu())
            all_probs.extend(probs.cpu().numpy())
    avg_loss = running_loss / len(loader.dataset)
    acc, prec, rec, f1, sens, spec = compute_metrics(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        cmap=plt.cm.Blues, xticks_rotation=45, ax=ax
    )
    plt.title('Confusion Matrix - EfficientNet-B3 + CBAM')
    plt.tight_layout()
    plt.show()
    print('\nClassification Report')
    print(classification_report(all_labels, all_preds,
                                target_names=class_names, zero_division=0))
    n_classes  = len(class_names)
    y_true_bin = label_binarize(all_labels, classes=range(n_classes))
    y_score    = np.array(all_probs)
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    all_fpr  = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = sum(np.interp(all_fpr, fpr[i], tpr[i]) for i in range(n_classes)) / n_classes
    fpr['macro'], tpr['macro'], roc_auc['macro'] = all_fpr, mean_tpr, auc(all_fpr, mean_tpr)
    plt.figure(figsize=(10, 7))
    colors = plt.cm.tab10.colors
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], color=colors[i % len(colors)],
                 label=f'{class_names[i]} (AUC={roc_auc[i]:.2f})')
    plt.plot(fpr['macro'], tpr['macro'], 'k--', linewidth=2,
             label=f'Macro-avg (AUC={roc_auc["macro"]:.2f})')
    plt.plot([0,1],[0,1],'--', color='gray', linewidth=0.8)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves - EfficientNet-B3 + CBAM')
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print('\nAUC scores:')
    for i in range(n_classes):
        print(f'  {class_names[i]}: {roc_auc[i]:.4f}')
    print(f'  Macro-average: {roc_auc["macro"]:.4f}')
    return avg_loss, acc, prec, rec, f1, all_labels, all_preds, all_images, torch.tensor(np.vstack(all_probs))


def plot_most_incorrect(incorrect, classes, n_images=9, normalize=True):
    rows = cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(22, 20))
    for i in range(min(rows * cols, len(incorrect))):
        ax = fig.add_subplot(rows, cols, i+1)
        image, true_label, probs = incorrect[i]
        image = image.permute(1, 2, 0)
        if normalize:
            image = normalize_image(image)
        true_prob = probs[true_label]
        incorrect_prob, incorrect_label = torch.max(probs, dim=0)
        ax.imshow(image.numpy())
        ax.set_title(
            f'True: {classes[true_label]}\n({true_prob:.3f})\n'
            f'Pred: {classes[incorrect_label]}\n({incorrect_prob:.3f})',
            fontsize=8
        )
        ax.axis('off')
    fig.subplots_adjust(hspace=0.6)
    plt.show()

In [ ]:
class_weights_tensor = weights_tensor.to(device)
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=0.1
)

In [ ]:
model.freeze_backbone()

opt = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)

print('=== WARMUP PHASE ===')
for epoch in range(5):
    tr = train_one_epoch(model, train_loader, opt, criterion, device)
    vl = validate_one_epoch(model, val_loader, criterion, device)
    print(f'[warmup {epoch+1}/5] loss {tr[0]:.4f} acc {tr[1]:.4f} | '
          f'val loss {vl[0]:.4f} acc {vl[1]:.4f}')

model.unfreeze_backbone()

opt = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(opt, mode='min', factor=0.1, patience=10)

best_loss = float('inf')
no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('\n=== FULL FINE-TUNING PHASE ===')
for epoch in range(100):
    tr = train_one_epoch(model, train_loader, opt, criterion, device)
    vl = validate_one_epoch(model, val_loader, criterion, device)

    history['train_loss'].append(tr[0])
    history['val_loss'].append(vl[0])
    history['train_acc'].append(tr[1])
    history['val_acc'].append(vl[1])

    scheduler.step(vl[0])

    print(f'[{epoch+1:03d}] lr={opt.param_groups[0]["lr"]:.0e} | '
          f'loss {tr[0]:.4f}/{vl[0]:.4f} | acc {tr[1]:.4f}/{vl[1]:.4f} | '
          f'f1 {tr[4]:.4f}/{vl[4]:.4f} | sens {tr[5]:.4f}/{vl[5]:.4f}')

    if vl[0] < best_loss:
        best_loss = vl[0]
        no_improve = 0
        torch.save(model.state_dict(), 'best_efficientnet_b3_cbam_100X.pth')
        print(f'  saved best model (val_loss={best_loss:.4f})')
    else:
        no_improve += 1
        if no_improve >= 40:
            print(f'Early stopping at epoch {epoch+1}')
            break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'],   label='val')
ax1.set(title='Loss - EfficientNet-B3 + CBAM', xlabel='epoch')
ax1.legend(); ax1.grid(True)
ax2.plot([a*100 for a in history['train_acc']], label='train')
ax2.plot([a*100 for a in history['val_acc']],   label='val')
ax2.set(title='Accuracy % - EfficientNet-B3 + CBAM', xlabel='epoch')
ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load('best_efficientnet_b3_cbam_100X.pth'))

test_loss, test_acc, test_prec, test_rec, test_f1, \
    all_labels, all_preds, all_images, all_probs = test_model(
        model, test_loader, criterion, device, class_names
    )

corrects = torch.eq(torch.tensor(all_labels), torch.tensor(all_preds))
n_correct = corrects.sum().item()
print(f'\nTest results - EfficientNet-B3 + CBAM')
print(f'loss {test_loss:.4f} | acc {test_acc:.4f} | '
      f'prec {test_prec:.4f} | rec {test_rec:.4f} | f1 {test_f1:.4f}')
print(f'correct: {n_correct}/{len(corrects)} ({100*n_correct/len(corrects):.2f}%)')

incorrect = [
    (img, lbl, prob)
    for img, lbl, prob, ok in zip(all_images, all_labels, all_probs, corrects)
    if not ok
]
incorrect.sort(reverse=True, key=lambda x: torch.max(x[2]).item())
plot_most_incorrect(incorrect, class_names, n_images=9)

In [ ]:

def visualise_cbam_attention(model, loader, class_names, device, n_images=6):
    model.eval()

    cbam_blocks = [m for m in model.features.modules() if isinstance(m, CBAM)]
    if not cbam_blocks:
        print('No CBAM blocks found.')
        return
    print(f'Found {len(cbam_blocks)} CBAM blocks.')

    attention_maps = {i: [] for i in range(len(cbam_blocks))}
    hooks = []

    def make_hook(idx):
        def hook_fn(module, input, output):
            with torch.no_grad():
                avg_out = torch.mean(input[0], dim=1, keepdim=True)
                max_out, _ = torch.max(input[0], dim=1, keepdim=True)
                combined = torch.cat([avg_out, max_out], dim=1)
                attn = torch.sigmoid(module.spatial_attention.conv(combined))
                attention_maps[idx].append(attn.detach().cpu())
        return hook_fn

    for i, cbam in enumerate(cbam_blocks):
        hooks.append(cbam.register_forward_hook(make_hook(i)))

    images, labels = next(iter(loader))
    images_dev = images.to(device)

    with torch.no_grad():
        outputs = model(images_dev)
        preds   = torch.argmax(outputs, dim=1).cpu().numpy()

    for h in hooks:
        h.remove()

    n_show  = min(n_images, len(images))
    n_cbams = len(cbam_blocks)
    fig, axes = plt.subplots(n_show, n_cbams + 1,
                              figsize=(5 * (n_cbams + 1), 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for row in range(n_show):
        img = images[row].permute(1, 2, 0).numpy()
        img = normalize_image(torch.tensor(img)).numpy()
        axes[row, 0].imshow(img)
        axes[row, 0].set_title(
            f'Input\nTrue: {class_names[labels[row]]}\nPred: {class_names[preds[row]]}',
            fontsize=9
        )
        axes[row, 0].axis('off')

        for col_idx in range(n_cbams):
            attn_map = attention_maps[col_idx][0][row, 0]
            attn_up = F.interpolate(
                attn_map.unsqueeze(0).unsqueeze(0),
                size=(IMG_SIZE, IMG_SIZE),
                mode='bilinear',
                align_corners=False
            ).squeeze().numpy()
            axes[row, col_idx + 1].imshow(img)
            axes[row, col_idx + 1].imshow(attn_up, alpha=0.55, cmap='jet')
            axes[row, col_idx + 1].set_title(
                f'CBAM {col_idx + 1} Spatial Attention\n(after block {6 + col_idx})',
                fontsize=9
            )
            axes[row, col_idx + 1].axis('off')

    plt.suptitle('CBAM Spatial Attention - red=high attention, blue=low attention',
                  fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()


visualise_cbam_attention(model, test_loader, class_names, device, n_images=6)